# Linked Lists — Pointer-Chained Nodes

A **linked list** stores each element in a **node** holding a value and a reference to the next node, so the elements need not be contiguous in memory. This removes the array's shifting cost — insertion and deletion are $O(1)$ *once the spot is found* — but destroys random access: reaching index $i$ requires walking $i$ links from the head. Every operation below records a snapshot so the pointer rewiring can be replayed and watched.

$$ \text{access}(i) = O(n), \qquad \text{insert/delete at known node} = O(1). $$

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.figsize'] = (9, 3)

def make_player(n_steps, render, label='step'):
    slider = widgets.IntSlider(value=0, min=0, max=n_steps-1, description=label,
                               continuous_update=False, layout=widgets.Layout(width='60%'))
    play = widgets.Play(value=0, min=0, max=n_steps-1, interval=600)
    widgets.jslink((play, 'value'), (slider, 'value'))
    out = widgets.interactive_output(render, {'k': slider})
    display(widgets.HBox([play, slider]), out)

# Draw a list given as a sequence of (value, color) and optional cursor / dangling node.
def draw_list(values, colors, title, cursor=None, floating=None):
    n = len(values)
    fig, ax = plt.subplots(figsize=(max(8, 1.6*n), 3))
    ax.text(-0.9, 0.0, 'head', ha='center', va='center', fontsize=10, color='seagreen')
    for i, (v, c) in enumerate(zip(values, colors)):
        x = i*1.6
        ax.add_patch(plt.Rectangle((x, -0.35), 1.0, 0.7, facecolor=c, edgecolor='k'))
        ax.text(x+0.5, 0.0, str(v), ha='center', va='center', fontsize=12)
        if i < n-1:
            ax.annotate('', xy=(x+1.6, 0.0), xytext=(x+1.0, 0.0),
                        arrowprops=dict(arrowstyle='->', lw=1.6))
        else:
            ax.text(x+1.35, 0.0, 'None', ha='center', va='center', fontsize=8, color='gray')
        if cursor == i:
            ax.annotate('cur', xy=(x+0.5, 0.55), ha='center', color='tomato', fontsize=10)
    if floating is not None:
        fv, fc = floating
        ax.add_patch(plt.Rectangle((0, -1.4), 1.0, 0.7, facecolor=fc, edgecolor='k'))
        ax.text(0.5, -1.05, str(fv), ha='center', va='center', fontsize=12)
        ax.text(0.5, -1.7, 'new node', ha='center', va='center', fontsize=8, color='tomato')
    ax.set_xlim(-1.6, max(2, n*1.6)); ax.set_ylim(-2.0, 1.0)
    ax.set_title(title); ax.axis('off'); plt.show()

## Accessing Index $i$ Means Walking the Chain

There is no address arithmetic: to read position $i$ the algorithm starts at `head` and follows `next` exactly $i$ times. Cost grows linearly with the index, the opposite of an array. Step through to watch the cursor hop link by link toward the target.

$$ \text{steps to reach index } i = i. $$

In [2]:
base = [10, 20, 30, 40, 50, 60]

def access_frames(values, target):
    frames = []
    for i in range(len(values)):
        frames.append((i, i == target))
        if i == target: break
    return frames

tgt_s = widgets.IntSlider(value=4, min=0, max=len(base)-1, description='index i')
acc_area = widgets.Output()

def relaunch_acc(*_):
    frames = access_frames(base, tgt_s.value)
    def draw(k):
        cur, hit = frames[k]
        colors = ['seagreen' if hit and i==cur else ('tomato' if i==cur else 'lightsteelblue')
                  for i in range(len(base))]
        msg = f'arrived: a[{cur}] = {base[cur]}' if hit else f'hop to node {cur}'
        draw_list(base, colors, f'step {k}/{len(frames)-1}: {msg}', cursor=cur)
    acc_area.clear_output(wait=True)
    with acc_area: make_player(len(frames), draw)

tgt_s.observe(relaunch_acc, 'value')
display(tgt_s, acc_area)
relaunch_acc()

IntSlider(value=4, description='index i', max=5)

Output()

## Insertion Rewires Two Pointers

To insert a new node at position $k$, the list is walked to node $k-1$, then two pointer assignments splice the node in: the new node points to the successor, and the predecessor points to the new node. The walk is $O(k)$ but the rewiring itself is $O(1)$ — no elements move.

$$ \text{new.next} \leftarrow \text{pred.next}, \qquad \text{pred.next} \leftarrow \text{new}. $$

In [3]:
def insert_frames(values, k, val):
    frames = []
    # walk to predecessor
    for i in range(k):
        frames.append(('walk', list(values), i, val, f'walk to node {i}'))
    frames.append(('arrive', list(values), max(k-1,0), val, f'reached insert point (before index {k})'))
    frames.append(('point', list(values), max(k-1,0), val, 'new.next -> successor'))
    new_values = values[:k] + [val] + values[k:]
    frames.append(('linked', new_values, k, val, 'pred.next -> new node'))
    return frames

ins_vals = [10, 20, 30, 40]
ik_s = widgets.IntSlider(value=2, min=0, max=len(ins_vals), description='insert at k')
iv_s = widgets.IntSlider(value=99, min=1, max=99, step=1, description='value')
ins_area = widgets.Output()

def relaunch_ins(*_):
    frames = insert_frames(ins_vals, ik_s.value, iv_s.value)
    def draw(k):
        phase, vals, cur, nv, note = frames[k]
        if phase in ('walk', 'arrive', 'point'):
            colors = ['tomato' if i==cur else 'lightsteelblue' for i in range(len(vals))]
            draw_list(vals, colors, f'step {k}/{len(frames)-1}: {note}',
                      cursor=cur, floating=(nv, 'gold'))
        else:
            colors = ['gold' if vals[i]==nv and i==ik_s.value else 'lightsteelblue' for i in range(len(vals))]
            draw_list(vals, colors, f'step {k}/{len(frames)-1}: {note}', cursor=ik_s.value)
    ins_area.clear_output(wait=True)
    with ins_area: make_player(len(frames), draw)

ik_s.observe(relaunch_ins, 'value'); iv_s.observe(relaunch_ins, 'value')
display(widgets.HBox([ik_s, iv_s]), ins_area)
relaunch_ins()

Output()

## Deletion Bypasses a Node

Removing the node at position $k$ requires the predecessor to point past it: `pred.next ← target.next`. The removed node is then unreachable (garbage-collected). Again the walk dominates the cost while the unlink is a single $O(1)$ assignment.

$$ \text{pred.next} \leftarrow \text{target.next}. $$

In [4]:
def delete_frames(values, k):
    frames = []
    for i in range(k):
        frames.append(('walk', list(values), i, f'walk to node {i}'))
    frames.append(('target', list(values), k, f'target = node {k} (value {values[k]})'))
    frames.append(('bypass', list(values), k, 'pred.next -> target.next'))
    new_values = values[:k] + values[k+1:]
    frames.append(('done', new_values, max(k-1,0), f'node {k} unlinked'))
    return frames

del_vals = [10, 20, 30, 40, 50]
dk_s = widgets.IntSlider(value=2, min=0, max=len(del_vals)-1, description='delete k')
del_area = widgets.Output()

def relaunch_del(*_):
    frames = delete_frames(del_vals, dk_s.value)
    def draw(k):
        phase, vals, cur, note = frames[k]
        if phase == 'target':
            colors = ['tomato' if i==cur else 'lightsteelblue' for i in range(len(vals))]
        elif phase == 'bypass':
            colors = ['lightgray' if i==cur else 'lightsteelblue' for i in range(len(vals))]
        else:
            colors = ['tomato' if i==cur else 'lightsteelblue' for i in range(len(vals))]
        draw_list(vals, colors, f'step {k}/{len(frames)-1}: {note}', cursor=cur)
    del_area.clear_output(wait=True)
    with del_area: make_player(len(frames), draw)

dk_s.observe(relaunch_del, 'value')
display(dk_s, del_area)
relaunch_del()

IntSlider(value=2, description='delete k', max=4)

Output()

## Search Walks Until a Match

Finding a value means traversing from the head comparing each node, returning the position of the first match or failing after $n$ comparisons. There is no shortcut even if the list is sorted, because a linked list cannot jump to its middle in $O(1)$ — binary search is unavailable here.

$$ \text{comparisons} = \begin{cases} k+1 & \text{value at index } k \\ n & \text{absent}. \end{cases} $$

In [5]:
search_vals = [42, 17, 88, 5, 63, 29]

def search_frames(values, target):
    frames = []
    for i, v in enumerate(values):
        hit = (v == target)
        frames.append((i, hit))
        if hit: break
    else:
        frames.append((len(values), False))   # absent marker
    return frames

st_s = widgets.Dropdown(options=search_vals + [100], value=63, description='target')
srch_area = widgets.Output()

def relaunch_srch(*_):
    frames = search_frames(search_vals, st_s.value)
    def draw(k):
        cur, hit = frames[k]
        if cur >= len(search_vals):
            colors = ['lightgray']*len(search_vals)
            draw_list(search_vals, colors, f'step {k}/{len(frames)-1}: not found (scanned all {len(search_vals)})')
        else:
            colors = ['seagreen' if hit and i==cur else ('tomato' if i==cur else 'lightsteelblue')
                      for i in range(len(search_vals))]
            msg = f'FOUND at index {cur}' if hit else f'compare node {cur} ({search_vals[cur]}) != {st_s.value}'
            draw_list(search_vals, colors, f'step {k}/{len(frames)-1}: {msg}', cursor=cur)
    srch_area.clear_output(wait=True)
    with srch_area: make_player(len(frames), draw)

st_s.observe(relaunch_srch, 'value')
display(st_s, srch_area)
relaunch_srch()

Dropdown(description='target', index=4, options=(42, 17, 88, 5, 63, 29, 100), value=63)

Output()

## In-Place Reversal — The Three-Pointer Dance

Reversing a list without extra storage uses three pointers — `prev`, `cur`, `next` — that march down the chain flipping each link backward in turn. It is a single $O(n)$ pass with $O(1)$ extra space and a canonical pointer-manipulation exercise. Step through to watch each `next` link flip and the three pointers advance.

$$ \text{tmp} \leftarrow \text{cur.next};\quad \text{cur.next} \leftarrow \text{prev};\quad \text{prev} \leftarrow \text{cur};\quad \text{cur} \leftarrow \text{tmp}. $$

In [6]:
rev_vals = [1, 2, 3, 4, 5]

def reverse_frames(values):
    # Represent state as ordered list of node-values with each node's current 'next' target.
    nodes = list(values)
    nxt = {i: (i+1 if i+1 < len(nodes) else None) for i in range(len(nodes))}
    prev = None; cur = 0
    frames = [(dict(nxt), prev, cur, None, 'start: prev=None, cur=head')]
    while cur is not None:
        tmp = nxt[cur]
        nxt[cur] = prev                              # flip this link
        frames.append((dict(nxt), prev, cur, tmp, f'flip node {cur}.next -> {prev}'))
        prev = cur; cur = tmp
        frames.append((dict(nxt), prev, cur, None, f'advance: prev={prev}, cur={cur}'))
    frames.append((dict(nxt), prev, cur, None, 'done: prev is new head'))
    return frames, nodes

frames_rev, rev_nodes = reverse_frames(rev_vals)

def draw_reverse(k):
    nxt, prev, cur, tmp, note = frames_rev[k]
    n = len(rev_nodes)
    fig, ax = plt.subplots(figsize=(max(8, 1.8*n), 3.2))
    xpos = {i: i*1.8 for i in range(n)}
    for i in range(n):
        x = xpos[i]
        c = 'tomato' if i == cur else ('gold' if i == prev else 'lightsteelblue')
        ax.add_patch(plt.Rectangle((x, -0.35), 1.0, 0.7, facecolor=c, edgecolor='k'))
        ax.text(x+0.5, 0.0, str(rev_nodes[i]), ha='center', va='center', fontsize=12)
    for i in range(n):                                # draw current next-links (may point backward)
        t = nxt[i]
        if t is None: continue
        x0 = xpos[i]+0.5; x1 = xpos[t]+0.5
        rad = -0.4 if t < i else 0.0
        ax.annotate('', xy=(x1, 0.42 if t<i else 0.0), xytext=(x0, 0.42 if t<i else 0.0),
                    arrowprops=dict(arrowstyle='->', lw=1.5,
                                    color='darkred' if t<i else 'black',
                                    connectionstyle=f'arc3,rad={rad}'))
    if cur is not None: ax.annotate('cur', xy=(xpos[cur]+0.5, -0.7), ha='center', color='tomato', fontsize=9)
    if prev is not None: ax.annotate('prev', xy=(xpos[prev]+0.5, -0.7), ha='center', color='goldenrod', fontsize=9)
    ax.set_xlim(-0.8, n*1.8); ax.set_ylim(-1.2, 1.0)
    ax.set_title(f'step {k}/{len(frames_rev)-1}: {note}'); ax.axis('off'); plt.show()

make_player(len(frames_rev), draw_reverse)

Output()